# Notebook 1 — Full Training + Stratified 3:7 Cross-Class Forget Split

**Experiment:** ReGUn benchmark with fixed 3:7 random cross-class forget split.

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*
(Gao, Unal, Rangamani, Zhu — AISTATS 2026)

**This notebook produces:**

| Stage | Output | Path |
|-------|--------|------|
| **A** | Environment setup | — |
| **B** | Configuration | — |
| **C** | Pre-trained model Θ_o on FULL training set | `checkpoints/regun/pre_train/<tag>.pt` |
| **D** | Stratified 3:7 split files (3 seeds) | `checkpoints/regun/splits/forget_indices_seed{s}.json` |
| **E** | Save `regun_config.json` | `checkpoints/regun/regun_config.json` |

> **After this notebook finishes:** publish `checkpoints/regun/` as a Kaggle dataset,
> then attach it to Notebooks 2, 3, and 4.

> Full mode: 300-epoch pretrain on T4 ≈ 2-3 h. Set `TEST_MODE=True` for a ~1 min check.

## A. Environment Setup

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## B. Configuration

| Mode | `TEST_MODE` | What it does | Time |
|------|------------|--------------|------|
| **Test** | `True` | 1% data, 1 epoch | ~1 min (CPU) |
| **Full** | `False` | Full data, 300-epoch pretrain | ~2-3 h (T4) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FLIP THIS to False for the real experiment run
TEST_MODE     = True
TEST_FRACTION = 0.01   # only used in TEST_MODE
# ══════════════════════════════════════════════════════════════════════

# ── Previous-run dataset (optional) ──────────────────────────────────
# If you have already run this notebook before, set this to skip re-training.
PREV_RUN_DATASET_DIR = None  # e.g. '/kaggle/input/regun-notebook1'

# ── Dataset / architecture ────────────────────────────────────────────
DATASET   = 'cifar10'    # 'cifar10' | 'cifar100' | 'tinyimagenet'
ARCH      = {'cifar10': 'resnet18', 'cifar100': 'resnet18',
              'tinyimagenet': 'resnet50'}[DATASET]
IS_VIT    = False
DATA_PATH = '/kaggle/working/data'

# ── Split seeds ───────────────────────────────────────────────────────
# Generate 3 independent stratified 3:7 cross-class forget splits.
# All downstream notebooks load these files — never regenerate.
SPLIT_SEEDS      = [0, 1, 2]
FORGET_FRACTION  = 0.30   # exactly 30% of EACH class -> forget set

# ── Dataset size constants ────────────────────────────────────────────
_TOTAL     = {'cifar10': 50000, 'cifar100': 50000, 'tinyimagenet': 100000}
_PER_CLASS = {'cifar10': 5000,  'cifar100': 500,   'tinyimagenet': 500}

# ── Pre-training hyperparameters ──────────────────────────────────────
if TEST_MODE:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 1, 8, 1
else:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 300, 128, 50

# ── Checkpoint output ─────────────────────────────────────────────────
CKPT_ROOT  = '/kaggle/working/checkpoints/regun'
SPLIT_DIR  = f'{CKPT_ROOT}/splits'
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

_MODE_TAG = 'test' if TEST_MODE else 'full'

print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Pretrain epochs={PRETRAIN_EPOCHS}  Forget fraction={FORGET_FRACTION}')
print(f'Split seeds: {SPLIT_SEEDS}')

## C-helpers. Args & Data Helpers

In [ ]:
from utils import get_dataset, get_model, test, load_encoder_ckpt_safely
from train import train as train_one_epoch
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=None, class_label_names=None,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='regun',
    )
    d.update(ov)
    return argparse.Namespace(**d)

def _find_ckpt(relative_path):
    local = f'{CKPT_ROOT}/{relative_path}'
    if os.path.exists(local):
        return local
    if PREV_RUN_DATASET_DIR:
        prev = f'{PREV_RUN_DATASET_DIR}/{relative_path}'
        if os.path.exists(prev):
            print(f'  [resume] found in dataset: {prev}')
            return prev
    return None

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
NUM_CLASSES       = args_base.num_classes
CLASS_LABEL_NAMES = args_base.class_label_names
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_targets = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_targets[i] for i in kept]
        return sub

    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: {TEST_FRACTION*100:.1f}% → '
          f'Train={len(dataset_train)}  Test={len(dataset_test)}')
else:
    print(f'Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2,
                 pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)
print(f'train_loader: {len(train_loader)} batches  test_loader: {len(test_loader)} batches')

## C. Pre-train Θ_o on Full Training Set

In [ ]:
CKPT_PRETRAIN = f'{CKPT_ROOT}/pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
os.makedirs(os.path.dirname(CKPT_PRETRAIN), exist_ok=True)

existing = _find_ckpt(f'pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt')
if existing:
    print(f'Pre-train checkpoint found: {existing} — skipping training.')
    CKPT_PRETRAIN = existing
else:
    print('Training Θ_o on full training set...')
    args_pt = make_args(
        unlearn_method='pre_train',
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR,
        patience=PRETRAIN_PATIENCE,
        num_classes=NUM_CLASSES,
        class_label_names=CLASS_LABEL_NAMES,
        remove_FC=True,
        CMFClassifier=True,
    )

    model = get_model(args_pt, device)
    optimizer = optim.SGD(
        model.parameters(), lr=PRETRAIN_LR,
        momentum=0.9, weight_decay=5e-4, nesterov=True
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=PRETRAIN_EPOCHS
    )

    best_acc = 0.0
    train_log = []
    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            # Recompute CMF weights from full train set each epoch (cmf_static pattern)
            if hasattr(model, 'recompute_cmf'):
                model.eval()
                model.recompute_cmf(train_loader, device=device)
                model.train()
                loss, acc = model.forward_a((x, y), stage='train')
            else:
                output = model(x)
                loss = torch.nn.functional.cross_entropy(output, y)
                acc = (output.argmax(1) == y).float().mean()
            loss.backward()
            optimizer.step()
            bs = x.size(0)
            total_loss += loss.item() * bs
            total_correct += int(acc.item() * bs)
            total_n += bs
        scheduler.step()
        if hasattr(model, 'recompute_cmf'):
            model.eval()
            model.recompute_cmf(train_loader, device=device)
        avg_loss = total_loss / max(1, total_n)
        avg_acc  = total_correct / max(1, total_n)
        train_log.append({'epoch': epoch, 'loss': avg_loss, 'acc': avg_acc})
        if epoch % max(1, PRETRAIN_EPOCHS // 10) == 0 or epoch == PRETRAIN_EPOCHS:
            print(f'  Epoch {epoch:3d}/{PRETRAIN_EPOCHS}  loss={avg_loss:.4f}  acc={avg_acc:.4f}')
            ra, fa, _ = test(model, device, test_loader, [],
                             CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')
            if ra > best_acc:
                best_acc = ra
                torch.save(model.state_dict(), CKPT_PRETRAIN)
                print(f'  → saved best checkpoint (acc={best_acc:.4f})')

    # Save final regardless
    torch.save(model.state_dict(), CKPT_PRETRAIN)
    # Save training log
    with open(f'{CKPT_ROOT}/pre_train/train_log.json', 'w') as f:
        json.dump(train_log, f, indent=2)
    print(f'Pre-train complete → {CKPT_PRETRAIN}')

print(f'\n── Final pre-trained model accuracy ──')
args_pt = make_args(unlearn_method='pre_train', num_classes=NUM_CLASSES,
                    class_label_names=CLASS_LABEL_NAMES, remove_FC=True, CMFClassifier=True)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
if hasattr(orig_model, 'recompute_cmf'):
    orig_model.recompute_cmf(train_loader, device=device)
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')

## D. Build Stratified 3:7 Cross-Class Forget Splits

`build_stratified_random_split` samples exactly 30% of EACH class's training samples
into the forget set and 70% into retain. This produces a cross-class forget set
(not whole-class removal). Saved as JSON index files for reproducibility.

In [ ]:
def build_stratified_random_split(dataset, forget_fraction=0.30, seed=0):
    """
    Stratified random cross-class forget split.

    Exactly `forget_fraction` of each class's samples go to the forget set;
    the rest go to retain. The forget set is randomly mixed across ALL classes
    (no whole-class removal). This gives an exact 3:7 ratio both overall and
    per-class.

    Args:
        dataset   : PyTorch dataset with .targets attribute
        forget_fraction : fraction of each class to forget (default 0.30)
        seed      : random seed for reproducibility

    Returns:
        forget_indices (list[int]), retain_indices (list[int])
    """
    # Collect indices per class
    rng = random.Random(seed)
    if hasattr(dataset, 'targets'):
        all_targets = dataset.targets
    elif hasattr(dataset, 'indices'):
        # Subset
        all_targets = [dataset.dataset.targets[i] for i in dataset.indices]
    else:
        all_targets = [dataset[i][1] for i in range(len(dataset))]

    by_class = collections.defaultdict(list)
    for idx, lbl in enumerate(all_targets):
        by_class[int(lbl)].append(idx)

    forget_indices = []
    retain_indices = []
    per_class_log  = {}

    for cls in sorted(by_class.keys()):
        pool = list(by_class[cls])
        rng.shuffle(pool)
        n_forget = round(len(pool) * forget_fraction)  # round for exact fraction
        n_forget = max(1, min(n_forget, len(pool) - 1))  # at least 1 in each set
        forget_indices.extend(pool[:n_forget])
        retain_indices.extend(pool[n_forget:])
        per_class_log[cls] = {
            'total': len(pool),
            'forget': n_forget,
            'retain': len(pool) - n_forget,
            'forget_pct': f'{100*n_forget/len(pool):.1f}%',
        }

    return forget_indices, retain_indices, per_class_log


# ── Generate 3 splits and save ────────────────────────────────────────
split_manifest = {}

for seed in SPLIT_SEEDS:
    forget_idx, retain_idx, per_class_log = build_stratified_random_split(
        dataset_train, forget_fraction=FORGET_FRACTION, seed=seed
    )

    out_path = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    payload = {
        'seed': seed,
        'forget_fraction': FORGET_FRACTION,
        'dataset': DATASET,
        'total_train': len(dataset_train),
        'n_forget': len(forget_idx),
        'n_retain': len(retain_idx),
        'forget_indices': forget_idx,
        'retain_indices': retain_idx,
        'per_class': per_class_log,
    }
    with open(out_path, 'w') as f:
        json.dump(payload, f)

    split_manifest[seed] = out_path

    # ── Per-class log confirming exact 30% ────────────────────────────
    print(f'\n── Seed {seed} split ─────────────────────────────────────')
    print(f'  Total train={len(dataset_train)}  forget={len(forget_idx)}  '
          f'retain={len(retain_idx)}  '
          f'overall_forget%={100*len(forget_idx)/len(dataset_train):.1f}%')
    print(f'  Per-class forget counts:')
    header = f'{"class":>6}  {"total":>7}  {"forget":>7}  {"retain":>7}  {"pct":>6}'
    print('  ' + header)
    print('  ' + '-' * len(header))
    for cls, info in sorted(per_class_log.items()):
        print(f'  {cls:>6}  {info["total"]:>7}  {info["forget"]:>7}  '
              f'{info["retain"]:>7}  {info["forget_pct"]:>6}')
    print(f'  → saved: {out_path}')

print(f'\nAll {len(SPLIT_SEEDS)} splits saved to {SPLIT_DIR}/')

## E. Save Config

In [ ]:
CONFIG = {
    # Mode
    'TEST_MODE': TEST_MODE,
    'TEST_FRACTION': TEST_FRACTION,
    '_MODE_TAG': _MODE_TAG,
    # Dataset / arch
    'DATASET': DATASET,
    'ARCH': ARCH,
    'IS_VIT': IS_VIT,
    'NUM_CLASSES': NUM_CLASSES,
    'CLASS_LABEL_NAMES': CLASS_LABEL_NAMES,
    'TOTAL': _TOTAL[DATASET],
    'PER_CLASS': _PER_CLASS[DATASET],
    # Splits
    'SPLIT_SEEDS': SPLIT_SEEDS,
    'FORGET_FRACTION': FORGET_FRACTION,
    'SPLIT_DIR': SPLIT_DIR,
    # Training
    'PRETRAIN_LR': PRETRAIN_LR,
    'PRETRAIN_EPOCHS': PRETRAIN_EPOCHS,
    'PRETRAIN_BS': PRETRAIN_BS,
    'PRETRAIN_PATIENCE': PRETRAIN_PATIENCE,
    # Checkpoints
    'CKPT_ROOT': CKPT_ROOT,
    'CKPT_PRETRAIN': CKPT_PRETRAIN,
}

config_path = f'{CKPT_ROOT}/regun_config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f'Config saved: {config_path}')
print(json.dumps({k: v for k, v in CONFIG.items() if not isinstance(v, list) or len(v) < 20},
                 indent=2))

## Summary

**Outputs ready for downstream notebooks:**

| File | Description |
|------|-------------|
| `checkpoints/regun/pre_train/<tag>.pt` | Θ_o — original model trained on FULL training set |
| `checkpoints/regun/splits/forget_indices_seed0.json` | Seed 0 split: 30% forget / 70% retain per class |
| `checkpoints/regun/splits/forget_indices_seed1.json` | Seed 1 split |
| `checkpoints/regun/splits/forget_indices_seed2.json` | Seed 2 split |
| `checkpoints/regun/regun_config.json` | All config for downstream notebooks |

**Next:** Publish `checkpoints/regun/` as a Kaggle dataset, then run Notebook 2 (oracle retrain).